###STEPPING SCHEMA

All discrete or instantaneous changes can be characterised as *flow fields*. An image of flow fields can be rapidly and efficiently transformed with `scipy.ndimage.warp_coordinates`, which can handle non-integer advection (and to a degree, follows backwards euler practice). This avoids the oscillatory errors particularly apparent in conventionally cellular transport using a forward euler integration.

The flow field itself is simple to determine, in the case of advection - however, over time and space, it becomes dampened. This is also true of diffusion, which is self-dampening on a local scale. In recognition of this fact, n-order secondary terms must either be implemented explicitly or implicitly included in function. Thus it becomes necessary to resolve over the spatial scale a 'smearing' of influence : i.e. the diffusion of flow itself, as effected by natural recursion.

Yet, where the difference has not yet been computed, how can the change be identified? Implicit Euler attempts to provide a solution to this problem; however, in terms of computing complexity, it by no means fully removed the computational burden imposed by simulating forward euler to a satisfactory precision. Neither does what I'm planning, but hell, it seems more flexible to me.

The flow field represents not just the instantaneous displacement of a parcel, but also the span of its historical presence: for a particle to travel through 5 metres of air, it must interact with 5 metres' worth of air. It is therefore reasonable that the summed movement of such an interaction can itself be observed as a parcel, in effect a window, over which the flow, temporally diffused, is observed in reality.

This 'diffusivity of flow' occurs in proportion with the magnitude of the flow - and mayhaps, where a gradient in flow exists, there is inevitably a measure of entropy that it represents. I thus hesitantly propose the determination of the flow-smoothing window to be occurrent and predictable through this second-order emergent process, as inspired by information theory.

In [4]:
import numpy as np
import scipy
import PhreeqcSolver

class StructGrid:

    def __init__(self):
        self.shape = (100, 30)
        self.grid_res = np.array([0.01, 0.01]) #1x1 cm grid cells
        self.param_len = 10
        self.property_len = 5
        self.struct = np.zeros((self.shape[0], self.shape[1], self.param_len, self.property_len))
        #IND    GRIDSIZE       PARAM      PROPERTY
        #0   DEPTH .. WIDTH .. salt   .. ??
        #1         ..       .. water  .. concentration
        #2         ..       .. ice    .. ??
        #3         ..       ..        .. veli (dim0)
        #4         ..       ..        .. velj (dim1)

        self.ist = 260
        self.ibt = 271.35
        self.temperature = np.full(self.shape, self.ibt)
        self.init_phase_solver()

    def init_phase_solver(self):
        phase_solver = PhreeqcSolver.frezchem_seawater()
        temps = np.linspace(self.ist, self.ibt, self.shape[0])
        salis = np.linspace(0, np.max(self.struct[:,:,0,1]), self.shape[0])
        ifrac = np.zeros((self.shape[0], self.shape[0]))
        for i in range(self.shape[0]):
            for j in range(self.shape[0]):
                ifrac[i,j] = phase_solver.solve_at(temps[i],salis[j])[0]
        ifrac = np.clip(ifrac[i,j], 0, 1)
        ifrac = scipy.interpolate.RectBivariateSpline(temps, salis, ifrac)
        self.ifrac_ev = ifrac.ev
        
    def update_state_field(self):
        # update temperature
        for j in range(self.shape[1]):
            mask = self.struct[:,j,2,1] > 1e-6 #1d mask
            self.temperature[:, mask] = np.linspace(self.ist, self.ibt, mask.size)

        # update phase
        delI = self.ifrac_ev(self.temperature.ravel(), (self.struct[:,:,0,1]/self.struct[:,:,1,1]).ravel()).reshape(self.shape)
        self.struct[:,:,1,1] = self.struct[:,:,2,1] + self.struct[:,:,1,1]
    
    def diffuse_over_flow(self, deltime):
        shift = self.struct[:,:,:,3:4] / self.grid_res[np.newaxis, np.newaxis, np.newaxis, :]
        
        
        pass

    def calculate_flow(self):
        pass
            
        
        
        